# Waste AI — Entraînement amélioré v2
**TrashNet + TACO + Augmentation forte + Fine-tuning complet**

Exécute les cellules dans l'ordre. GPU T4 requis.

## Étape 1 — Installation

In [ ]:
!pip install torch torchvision matplotlib scikit-learn seaborn albumentations pycocotools requests -q

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE != 'cuda':
    print('ATTENTION : Active le GPU T4 -> Exécution > Modifier le type d execution > GPU')

## Étape 2 — Téléchargement TrashNet

In [ ]:
import os

if not os.path.exists('trashnet/dataset-resized'):
    print('Téléchargement TrashNet...')
    !wget -q https://github.com/garythung/trashnet/raw/master/data/dataset-resized.zip
    !unzip -q dataset-resized.zip -d trashnet/
    print('TrashNet téléchargé')
else:
    print('TrashNet déjà présent')

print('Classes TrashNet :', sorted(os.listdir('trashnet/dataset-resized')))

## Étape 3 — Téléchargement TACO

In [ ]:
import os

if not os.path.exists('TACO'):
    print('Clonage TACO...')
    !git clone --quiet https://github.com/pedropro/TACO.git
    print('Repo cloné')
else:
    print('TACO repo déjà présent')

!pip install pycocotools requests -q
print('Dépendances TACO installées')

In [ ]:
import os

# Télécharger les images TACO (10-30 min selon la connexion)
if not os.path.exists('TACO/data/annotations.json'):
    print('Téléchargement des images TACO — patientez (10-30 min)...')
    os.chdir('TACO')
    !python download.py
    os.chdir('/content')
else:
    print('TACO déjà téléchargé')

print('Vérification :', os.path.exists('TACO/data/annotations.json'))

## Étape 4 — Organisation des images TACO par classe

In [ ]:
import os, json, shutil

TACO_TO_CLASS = {
    'Plastic bottle': 'plastic', 'Plastic bag & wrapper': 'plastic',
    'Plastic container': 'plastic', 'Plastic cup': 'plastic',
    'Plastic lid': 'plastic', 'Plastic straw': 'plastic',
    'Plastic utensils': 'plastic', 'Six pack rings': 'plastic',
    'Styrofoam piece': 'plastic',
    'Paper': 'paper', 'Paper bag': 'paper', 'Newspaper': 'paper',
    'Cardboard': 'cardboard', 'Carton': 'cardboard', 'Drink carton': 'cardboard',
    'Glass bottle': 'glass', 'Broken glass': 'glass', 'Glass jar': 'glass',
    'Aluminium foil': 'metal', 'Metal bottle cap': 'metal',
    'Drink can': 'metal', 'Food can': 'metal', 'Pop tab': 'metal',
    'Scrap metal': 'metal',
    'Cigarette': 'trash', 'Rope & strings': 'trash',
    'Shoe': 'trash', 'Unlabeled litter': 'trash',
}

CLASS_NAMES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
TACO_OUT = 'taco_organized'

for cls in CLASS_NAMES:
    os.makedirs(f'{TACO_OUT}/{cls}', exist_ok=True)

with open('TACO/data/annotations.json') as f:
    taco_data = json.load(f)

id_to_file = {img['id']: img['file_name'] for img in taco_data['images']}
cat_id_to_name = {cat['id']: cat['name'] for cat in taco_data['categories']}

count = 0
for ann in taco_data['annotations']:
    cat_name = cat_id_to_name.get(ann['category_id'], '')
    target_class = TACO_TO_CLASS.get(cat_name)
    if target_class is None:
        continue
    src = f"TACO/data/{id_to_file[ann['image_id']]}"
    dst = f"{TACO_OUT}/{target_class}/{ann['image_id']}_{ann['id']}.jpg"
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst)
        count += 1

print(f'TACO : {count} images organisées')
for cls in CLASS_NAMES:
    n = len(os.listdir(f'{TACO_OUT}/{cls}'))
    print(f'  {cls}: {n} images')

## Étape 5 — Dataset & Augmentation

In [ ]:
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, ConcatDataset, random_split
from torchvision import datasets
import albumentations as A
from albumentations.pytorch import ToTensorV2

CLASS_NAMES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
TACO_OUT = 'taco_organized'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

class WasteDataset(Dataset):
    def __init__(self, root, transform=None):
        self.data = datasets.ImageFolder(root)
        self.transform = transform
        self.classes = self.data.classes

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        path, label = self.data.samples[idx]
        image = np.array(Image.open(path).convert('RGB'))
        if self.transform:
            image = self.transform(image=image)['image']
        return image, label


train_transform = A.Compose([
    A.RandomResizedCrop(size=(224, 224), scale=(0.5, 1.0)),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.Rotate(limit=45, p=0.5),
    A.OneOf([
        A.GaussNoise(std_range=(0.05, 0.15)),
        A.GaussianBlur(blur_limit=3),
        A.MotionBlur(blur_limit=3),
    ], p=0.4),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3),
        A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30),
        A.CLAHE(clip_limit=4),
    ], p=0.5),
    A.CoarseDropout(num_holes_range=(1, 8), hole_height_range=(8, 32), hole_width_range=(8, 32), p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(height=224, width=224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

trashnet_ds = WasteDataset('trashnet/dataset-resized', transform=train_transform)
taco_ds = WasteDataset(TACO_OUT, transform=train_transform)

print(f'TrashNet : {len(trashnet_ds)} images | classes : {trashnet_ds.classes}')
print(f'TACO     : {len(taco_ds)} images | classes : {taco_ds.classes}')

full_dataset = ConcatDataset([trashnet_ds, taco_ds])
print(f'Dataset combiné : {len(full_dataset)} images')

n_train = int(0.8 * len(full_dataset))
n_val = len(full_dataset) - n_train
train_set, val_set = random_split(full_dataset, [n_train, n_val],
                                   generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train : {n_train} | Val : {n_val}')

## Étape 6 — Modèle MobileNetV3

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CLASS_NAMES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
NUM_CLASSES = len(CLASS_NAMES)

model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
model.classifier[3] = nn.Linear(model.classifier[3].in_features, NUM_CLASSES)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

print(f'Modèle prêt sur {DEVICE} — {NUM_CLASSES} classes')

## Étape 7 — Entraînement (Phase 1 : tête, Phase 2 : fine-tuning complet)

In [ ]:
import os

os.makedirs('checkpoints', exist_ok=True)
history = {'train_loss': [], 'val_acc': []}

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if train:
                optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            correct += (out.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), correct / total * 100


# Phase 1 — tête seulement (5 epochs)
print('Phase 1 : entraînement de la tête (5 epochs)')
for param in model.features.parameters():
    param.requires_grad = False

optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)

for epoch in range(5):
    loss, _ = run_epoch(train_loader, train=True)
    _, acc = run_epoch(val_loader, train=False)
    scheduler.step()
    history['train_loss'].append(loss)
    history['val_acc'].append(acc)
    print(f'  Epoch {epoch+1}/5 | Loss: {loss:.4f} | Val Acc: {acc:.1f}%')


# Phase 2 — fine-tuning complet (15 epochs)
print('\nPhase 2 : fine-tuning complet (15 epochs)')
for param in model.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

best_acc = 0
for epoch in range(15):
    loss, _ = run_epoch(train_loader, train=True)
    _, acc = run_epoch(val_loader, train=False)
    scheduler.step()
    history['train_loss'].append(loss)
    history['val_acc'].append(acc)
    print(f'  Epoch {epoch+1}/15 | Loss: {loss:.4f} | Val Acc: {acc:.1f}%')
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), 'checkpoints/waste_ai_v2.pt')
        print(f'    -> Meilleur modèle sauvegardé ({acc:.1f}%)')

print(f'\nMeilleure accuracy : {best_acc:.1f}%')

## Étape 8 — Courbes & Matrice de confusion

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Courbes
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history['train_loss'], color='#e74c3c')
ax1.axvline(x=4, color='gray', linestyle='--', label='Début phase 2')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.legend()
ax2.plot(history['val_acc'], color='#2ecc71')
ax2.axvline(x=4, color='gray', linestyle='--', label='Début phase 2')
ax2.set_title('Val Accuracy (%)')
ax2.set_xlabel('Epoch')
ax2.legend()
plt.suptitle('Waste AI v2 — TrashNet + TACO', fontsize=14)
plt.tight_layout()
plt.savefig('checkpoints/training_curves_v2.png', dpi=150)
plt.show()

# Matrice de confusion
CLASS_NAMES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
model.load_state_dict(torch.load('checkpoints/waste_ai_v2.pt'))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        preds = model(imgs.to(DEVICE)).argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Matrice de confusion — Waste AI v2')
plt.ylabel('Réel')
plt.xlabel('Prédit')
plt.tight_layout()
plt.savefig('checkpoints/confusion_matrix.png', dpi=150)
plt.show()

## Étape 9 — Télécharger le modèle

In [ ]:
from google.colab import files

files.download('checkpoints/waste_ai_v2.pt')
files.download('checkpoints/confusion_matrix.png')
files.download('checkpoints/training_curves_v2.png')

print('Téléchargements lancés !')
print('-> Place waste_ai_v2.pt dans waste-ai/model/checkpoints/')
print('-> L API chargera automatiquement la v2')